In [1]:
import pandas as pd


# =========================
# 1. CARGAR ARCHIVOS
# =========================

cursos = pd.read_excel(
    "formulario_limpio.xlsx",
    sheet_name="Cursos validados",
    dtype=str
)

catalogo = pd.read_excel(
    "formulario_limpio.xlsx",
    sheet_name="Catálogo VRAI",
    dtype=str
)

convenios = pd.read_excel(
    "Convenios.xlsx",
    dtype=str
)


# =========================
# 2. NORMALIZAR RUT UC
# =========================

def normalizar_rut(valor):
    if pd.isna(valor):
        return ""

    valor = str(valor).strip().upper()

    if valor.endswith(".0"):
        valor = valor[:-2]

    # Quitar puntos, guion y espacios
    valor = (
        valor
        .replace(".", "")
        .replace("-", "")
        .replace(" ", "")
    )

    return valor


cursos["_rut"] = cursos["RUT UC"].apply(normalizar_rut)
convenios["_rut"] = convenios["RUT UC"].apply(normalizar_rut)


# =========================
# 3. CREAR DICCIONARIO RUT -> ÁREA
# =========================

rut_area = (
    convenios[convenios["_rut"] != ""]
    .drop_duplicates(subset="_rut", keep="first")
    .set_index("_rut")["Area"]
    .to_dict()
)


# =========================
# 4. RELLENAR UA CONVENIO
# =========================

cursos["UA convenio"] = cursos["_rut"].map(rut_area)


# =========================
# 5. ELIMINAR COLUMNA AUXILIAR
# =========================

cursos = cursos.drop(columns=["_rut"])


# =========================
# 6. GUARDAR ARCHIVO FINAL
# =========================

with pd.ExcelWriter(
    "formulario_limpio_con_convenios.xlsx",
    engine="openpyxl"
) as writer:

    cursos.to_excel(
        writer,
        sheet_name="Cursos validados",
        index=False
    )

    catalogo.to_excel(
        writer,
        sheet_name="Catálogo VRAI",
        index=False
    )


print("Archivo guardado: formulario_limpio_con_convenios.xlsx")
print("Filas con UA convenio:", cursos["UA convenio"].notna().sum())

Archivo guardado: formulario_limpio_con_convenios.xlsx
Filas con UA convenio: 1960


In [1]:
import pandas as pd


# =========================
# 1. CARGAR ARCHIVOS
# =========================

catalogo = pd.read_excel(
    "formulario_limpio_1.xlsx",
    sheet_name="Catálogo VRAI",
    dtype=str
)

cursos_disponibles = pd.read_excel(
    "Cursos disponibles 2026-2.xlsx",
    dtype=str
)


# =========================
# 2. NORMALIZAR NRC Y SIGLA
# =========================

def normalizar_texto(valor):
    if pd.isna(valor):
        return ""
    return str(valor).strip().upper()


def normalizar_nrc(valor):
    if pd.isna(valor):
        return ""

    valor = str(valor).strip()

    if valor.endswith(".0"):
        valor = valor[:-2]

    return valor


catalogo["_nrc"] = catalogo["NRC"].apply(normalizar_nrc)
catalogo["_sigla"] = catalogo["Sigla"].apply(normalizar_texto)

cursos_disponibles["_nrc"] = cursos_disponibles["NRC"].apply(normalizar_nrc)
cursos_disponibles["_sigla"] = cursos_disponibles["Sigla"].apply(normalizar_texto)


# =========================
# 3. CREAR DICCIONARIOS
# =========================

reserva_por_nrc = (
    cursos_disponibles[
        cursos_disponibles["_nrc"] != ""
    ]
    .drop_duplicates("_nrc")
    .set_index("_nrc")["Reserva Máximo Programado"]
    .to_dict()
)

reserva_por_sigla = (
    cursos_disponibles[
        cursos_disponibles["_sigla"] != ""
    ]
    .drop_duplicates("_sigla")
    .set_index("_sigla")["Reserva Máximo Programado"]
    .to_dict()
)


# =========================
# 4. COMPLETAR VACANTES
# =========================

def buscar_vacantes(fila):

    # Primero intenta por NRC
    if fila["_nrc"] in reserva_por_nrc:
        return reserva_por_nrc[fila["_nrc"]]

    # Si no encuentra, intenta por SIGLA
    if fila["_sigla"] in reserva_por_sigla:
        return reserva_por_sigla[fila["_sigla"]]

    return pd.NA


catalogo["Vacantes Ofrecidas Máximas"] = catalogo.apply(
    buscar_vacantes,
    axis=1
)


# =========================
# 5. ELIMINAR AUXILIARES
# =========================

catalogo = catalogo.drop(
    columns=["_nrc", "_sigla"]
)


# =========================
# 6. GUARDAR RESULTADO
# =========================

catalogo.to_excel(
    "catalogo_vrai_con_vacantes.xlsx",
    index=False
)

print("Archivo guardado correctamente.")
print(
    "Filas con vacantes:",
    catalogo["Vacantes Ofrecidas Máximas"].notna().sum()
)

Archivo guardado correctamente.
Filas con vacantes: 872


In [1]:
import pandas as pd


# =========================
# 1. ARCHIVOS
# =========================

archivo_origen = "formulario_limpio.xlsx"
archivo_destino = "formulario_limpio_1.xlsx"
archivo_salida = "formulario_limpio_1_con_prioridad.xlsx"

hoja = "Cursos validados"


# =========================
# 2. CARGAR ARCHIVOS
# =========================

origen = pd.read_excel(
    archivo_origen,
    sheet_name=hoja,
    dtype=str
)

# Leer todas las hojas del archivo que será actualizado
hojas_destino = pd.read_excel(
    archivo_destino,
    sheet_name=None,
    dtype=str
)

destino = hojas_destino[hoja].copy()


# =========================
# 3. NORMALIZAR RUT UC
# =========================

def normalizar_rut(valor):
    if pd.isna(valor):
        return ""

    valor = str(valor).strip().upper()

    if valor.endswith(".0"):
        valor = valor[:-2]

    return (
        valor
        .replace(".", "")
        .replace("-", "")
        .replace(" ", "")
    )


origen["_rut"] = origen["RUT UC"].apply(normalizar_rut)
destino["_rut"] = destino["RUT UC"].apply(normalizar_rut)


# =========================
# 4. NUMERAR REPETICIONES DE CADA RUT
# =========================

origen["_orden_rut"] = origen.groupby("_rut").cumcount()
destino["_orden_rut"] = destino.groupby("_rut").cumcount()


# =========================
# 5. CREAR CLAVE RUT + ORDEN
# =========================

origen["_clave"] = (
    origen["_rut"]
    + "|"
    + origen["_orden_rut"].astype(str)
)

destino["_clave"] = (
    destino["_rut"]
    + "|"
    + destino["_orden_rut"].astype(str)
)


# =========================
# 6. ASIGNAR PRIORIDAD
# =========================

prioridad_por_clave = (
    origen
    .set_index("_clave")["Prioridad"]
    .to_dict()
)

destino["Prioridad"] = destino["_clave"].map(prioridad_por_clave)


# =========================
# 7. ELIMINAR COLUMNAS AUXILIARES
# =========================

destino = destino.drop(
    columns=["_rut", "_orden_rut", "_clave"]
)

hojas_destino[hoja] = destino


# =========================
# 8. GUARDAR TODAS LAS HOJAS
# =========================

with pd.ExcelWriter(
    archivo_salida,
    engine="openpyxl"
) as writer:

    for nombre_hoja, dataframe in hojas_destino.items():
        dataframe.to_excel(
            writer,
            sheet_name=nombre_hoja,
            index=False
        )


print("Archivo guardado:", archivo_salida)
print("Filas con Prioridad:", destino["Prioridad"].notna().sum())
print("Filas sin Prioridad:", destino["Prioridad"].isna().sum())

Archivo guardado: formulario_limpio_1_con_prioridad.xlsx
Filas con Prioridad: 3354
Filas sin Prioridad: 13
